# Google Drive Import 


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# mounting the google drive

# Import lib


In [ ]:
# ==========================
# Standard Library
# ==========================
import os

# ==========================
# TensorFlow
# ==========================
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import Model
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Reshape,
    concatenate,
    multiply
)

# Mixed Precision and GPU initialization 

In [ ]:
from tensorflow.keras import mixed_precision

# Force the GPU to use Tensor Cores for faster, lighter math
mixed_precision.set_global_policy('mixed_float16')
print("✅ Mixed precision activated!")

In [ ]:
import tensorflow as tf

# Check if TensorFlow sees the GPU
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f" GPU is LOCKED ON: {gpu_devices}")
    # Show detailed GPU info
    !nvidia-smi
else:
    print(" WARNING: No GPU detected. TensorFlow is using the CPU!")

# Dataloading and Copying into the run time cloud environment 

In [ ]:

tomato_dir=r"/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_"
print(f"Checking contents of: {tomato_dir}")

# List contents of the base dataset path (e.g., train, validation, test folders)
if os.path.exists(tomato_dir):
    print(f"Contents found at {tomato_dir}:")
    for item in os.listdir(tomato_dir):
        item_path = os.path.join(tomato_dir, item)
        print(f"- {item} {'(Directory)' if os.path.isdir(item_path) else '(File)'}")
else:
    print(f"Base dataset path does not exist: {tomato_dir}")


In [ ]:
train_dir = os.path.join(tomato_dir, 'train') # Assuming 'train' is the directory with class folders


print(f"Folders in '{train_dir}':")

# List all entries in the train_dir
for item in os.listdir(train_dir):
    item_path = os.path.join(train_dir, item)
    # Check if the item is a directory (a class folder)
    if os.path.isdir(item_path):
        print(f"- {item}")

In [ ]:



if os.path.exists(train_dir):
    print(f"\nCounting files in each class directory within: {train_dir}")
    class_counts = {}
    for class_name in os.listdir(train_dir):
        class_path = os.path.join(train_dir, class_name)
        if os.path.isdir(class_path):
            # Count only image files (e.g., .jpg, .jpeg, .png)
            num_files = len([name for name in os.listdir(class_path) if name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])
            class_counts[class_name] = num_files

    # Sort for consistent output
    sorted_class_counts = sorted(class_counts.items())

    for class_name, count in sorted_class_counts:
        print(f"{class_name}: {count} files")
else:
    print(f"The directory '{train_dir}' does not exist. Please check the dataset structure.")

In [ ]:

# this is for the  data integrity check to match with the original data set
def count_files_and_extensions(directory):
    total_files = 0
    extension_counts = {}

    for root, _, files in os.walk(directory):
        for file in files:
            total_files += 1
            ext = os.path.splitext(file)[1].lower()
            extension_counts[ext] = extension_counts.get(ext, 0) + 1

    return total_files, extension_counts

# Assuming base_dataset_path is already defined from previous cells
# base_dataset_path = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_'

if 'tomato_dir' in locals() and os.path.exists(tomato_dir):
    total, extensions = count_files_and_extensions(tomato_dir)
    print(f"Total files in dataset: {total}")
    print("File extension breakdown:")
    for ext, count in sorted(extensions.items()):
        print(f"  {ext}: {count}")
else:
    print(f"Base dataset path '{tomato_dir}' not found or not defined.")

In [ ]:
import shutil
import os
import tensorflow as tf

# 1. Define paths
drive_train_dir = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/train'
drive_valid_dir = '/content/drive/MyDrive/plant_disease_dataset/data_/Tomato_/valid'

local_train_dir = '/content/local_data/train'
local_valid_dir = '/content/local_data/valid'

# 2. Function to synchronize data from Drive to Local Runtime
def sync_local_data(src, dst):
    if os.path.exists(src):
        # Add a cleanup step: remove existing local directory if it exists
        if os.path.exists(dst):
            print(f"Removing existing local data at {dst}...")
            shutil.rmtree(dst)

        print(f"Copying {src} to local runtime...")
        shutil.copytree(src, dst)
    else:
        print(f"Source directory does not exist: {src}")

# Synchronize training data
sync_local_data(drive_train_dir, local_train_dir)

# Synchronize validation data
sync_local_data(drive_valid_dir, local_valid_dir)

# Main Model Building Starts
and

# DATA SET LOADING

In [ ]:
# we are intiating the Dense121 , a highly efficient convolutional neural network
#known for its connecting layer to every other layer

###  include_top=False,
#we chopped out the top layer don't want the model to output standard
#ImageNet classes (like "dog" or "car"); you want raw extracted features so you can add your own custom layers (like CondConv) on top.

 # weights="imagenet",
 #Instead of starting with random weights, the model initializes with weights learned from millions of images on the ImageNet dataset.
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

In [ ]:
# ----------------------------
# DATASET LOADING
# ----------------------------

batch_size = 32
img_height = 224
img_width = 224

print("Loading Training Dataset:")

train_ds = tf.keras.utils.image_dataset_from_directory(
    r'/content/local_data/train',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=True
)

print("\nLoading Validation Dataset:")

val_ds = tf.keras.utils.image_dataset_from_directory(
    r'/content/local_data/valid',
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical',
    shuffle=False
)

# ----------------------------
# PERFORMANCE SETTINGS
# ----------------------------

AUTOTUNE = tf.data.AUTOTUNE

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# ----------------------------
# PREPROCESS IMAGES
# ----------------------------

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

val_ds = val_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
)

# ----------------------------
# PIPELINE OPTIMIZATION
# ----------------------------

# train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)

# val_ds = val_ds.prefetch(AUTOTUNE)

# Cache the dataset in RAM (if it fits) and prefetch dynamically
train_ds = train_ds.prefetch(buffer_size=4)
val_ds = val_ds.prefetch(buffer_size=4)

print("✅ Safe Edge-Pipeline established. Ready for model.fit()")

# Dynamic Routing & Conditional Convolutions

In [ ]:
WEIGHT_DECAY = 2e-4

def conv2d(kernel_size, stride, filters, kernel_regularizer=tf.keras.regularizers.l2(WEIGHT_DECAY), padding="same", use_bias=False,
           kernel_initializer="he_normal", **kwargs):
    return layers.Conv2D(kernel_size=kernel_size, strides=stride, filters=filters, kernel_regularizer=kernel_regularizer, padding=padding,
                         use_bias=use_bias, kernel_initializer=kernel_initializer, **kwargs)

class Routing(layers.Layer):
    def __init__(self, out_channels, dropout_rate, temperature=30, **kwargs):
        super(Routing, self).__init__(**kwargs)
        self.avgpool = layers.GlobalAveragePooling2D()
        self.dropout = layers.Dropout(rate=dropout_rate)
        self.fc = layers.Dense(units=out_channels)
        self.softmax = layers.Softmax()
        self.temperature = temperature

    def call(self, inputs, **kwargs):
        out = self.avgpool(inputs)
        out = self.dropout(out)
        out = self.softmax(self.fc(out) * 1.0 / self.temperature)
        return out

class CondConv2D(layers.Layer):
    def __init__(self, filters, kernel_size, stride=1, use_bias=True, num_experts=3, padding="same", **kwargs):
        super(CondConv2D, self).__init__(**kwargs)
        self.routing = Routing(out_channels=num_experts, dropout_rate=0.2, name="routing_layer")
        self.convs = []
        for _ in range(num_experts):
            self.convs.append(conv2d(filters=filters, stride=stride, kernel_size=kernel_size, use_bias=use_bias, padding=padding))

    def call(self, inputs, **kwargs):
        routing_weights = self.routing(inputs)

        # MEMORY FIX: Reshape and broadcast instead of massive tf.transpose operations
        weight_0 = tf.reshape(routing_weights[:, 0], (-1, 1, 1, 1))
        feature = weight_0 * self.convs[0](inputs)

        for i in range(1, len(self.convs)):
            weight_i = tf.reshape(routing_weights[:, i], (-1, 1, 1, 1))
            feature += weight_i * self.convs[i](inputs)

        return feature

In [ ]:
kernel_init = tf.keras.initializers.glorot_uniform()
bias_init = tf.keras.initializers.Constant(value=0.0)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2), # Increased rotation
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1), # NEW: Drone camera jitter
    layers.RandomBrightness(factor=0.2), # NEW: Cloud cover/lighting changes
    layers.GaussianNoise(0.15) # Increased noise
], name="drone_condition_augmentation_v2")

In [ ]:
def Inception(x, nb_filter):
    # OPTIMIZATION: Replaced all nested CondConv2D calls with lightweight SeparableConv2D
    branch1x1 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(1,1), strides=1, padding='same', use_bias=True)(x)

    branch3x3 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(1,1), strides=1, padding='same', use_bias=True)(x)
    branch3x31 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(3,1), strides=1, padding='same', use_bias=True)(branch3x3)
    branch3x32 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(1,3), strides=1, padding='same', use_bias=True)(branch3x3)
    out1 = layers.Add()([branch3x31, branch3x32])

    branch5x5 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(1,1), strides=1, padding='same', use_bias=True)(x)
    branch5x5_1 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(3,1), strides=1, padding='same', use_bias=True)(branch5x5)
    branch5x5_2 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(1,3), strides=1, padding='same', use_bias=True)(branch5x5)
    out2 = layers.Add()([branch5x5_1, branch5x5_2])

    branch5x51 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(3,1), strides=1, padding='same', use_bias=True)(out2)
    branch5x52 = layers.SeparableConv2D(filters=nb_filter, kernel_size=(1,3), strides=1, padding='same', use_bias=True)(out2)
    out3 = layers.Add()([branch5x51, branch5x52])

    branchpool = layers.MaxPooling2D(pool_size=(3,3), strides=(1,1), padding='same')(x)
    branchpool = layers.SeparableConv2D(filters=nb_filter, kernel_size=(1,1), strides=1, padding='same', use_bias=True)(branchpool)

    x = layers.Concatenate(axis=3)([branch1x1, out1, out3, branchpool])
    return x

In [ ]:
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size
        })
        return config

    def call(self, images):
        # Print the input feature map shape
        print("=" * 60)
        print("INPUT TO PATCHES :", images.shape)

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )

        print("PATCH TENSOR :", patches.shape)

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [batch_size, -1, patch_dims]
        )

        print("PATCHES AFTER RESHAPE :", patches.shape)
        print("=" * 60)

        return patches

In [ ]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)

        self.num_patches = num_patches
        self.projection_dim = projection_dim

        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patch):

        print("=" * 60)
        print("INPUT PATCHES :", patch.shape)

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        print("NUMBER OF POSITIONS :", positions.shape)

        projected = self.projection(patch)

        print("PROJECTED PATCHES :", projected.shape)

        embedded = self.position_embedding(positions)

        print("POSITION EMBEDDING :", embedded.shape)

        output = projected + embedded

        print("FINAL OUTPUT :", output.shape)
        print("=" * 60)

        return output

In [ ]:
def sse_block(input_feature, ratio=4):
    """Implementation of Squeeze-and-Excitation(SE) block using Keras Layers."""
    channel_axis = -1
    channel = input_feature.shape[channel_axis]

    # Squeeze & Excitation Path
    se_feature = layers.GlobalAveragePooling2D()(input_feature)
    se_feature = layers.Reshape((1, 1, channel))(se_feature)
    se_feature = layers.Dense(channel // ratio, activation='relu', kernel_initializer='he_normal')(se_feature)
    se_feature = layers.Dense(channel, activation='sigmoid', kernel_initializer='he_normal')(se_feature)

    # Multiply the input by the SE features
    se_out = layers.Multiply()([input_feature, se_feature])

    # Spatial Statistics (Mean, Std, Max)
    # We wrap tf functions in Lambda layers to avoid the KerasTensor error
    mean = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(input_feature)
    std = layers.Lambda(lambda x: tf.math.reduce_std(x, axis=-1, keepdims=True))(input_feature)
    maximum = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(input_feature)

    # Final Concatenation
    out = layers.Concatenate()([se_out, mean, std, maximum])

    return out

In [ ]:
print("="*60)
print("REBUILDING CONDCONVIT: FROZEN BACKBONE & HIGH DROPOUT (0.4)")
print("="*60)

# --- 1. SET UP HYPERPARAMETERS ---
image_size = 28  
patch_size = 7   
num_patches = (image_size // patch_size) ** 2 
projection_dim = 32
num_heads = 2
transformer_units = [projection_dim * 2, projection_dim]
transformer_layers = 2
TOTAL_CLASSES = 11
HIGH_DROPOUT = 0.4 # <-- PHASE 2: Forcing generalization

# --- 2. AUGMENTED GRAPH & FROZEN BACKBONE ---
inputs = tf.keras.Input(shape=(224, 224, 3))
augmented_x = data_augmentation(inputs)

base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_tensor=augmented_x
)
base_model.trainable = False # CRITICAL: Stable, frozen backbone

# --- 3. MULTI-SCALE FEATURE EXTRACTION ---
x1 = base_model.get_layer('expanded_conv_project_BN').output
x1 = CondConv2D(kernel_size=3, filters=16, stride=2, padding='same', num_experts=3)(x1)
x1 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x1)
x1 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x1)
x1 = layers.Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x1)
x_conv1 = sse_block(x1, ratio=4)

x2 = base_model.get_layer('block_2_project_BN').output
x2 = CondConv2D(kernel_size=3, filters=16, stride=2, padding='same', num_experts=3)(x2)
x2 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x2)
x2 = layers.Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x2)
x_conv2 = sse_block(x2, ratio=4)

x_inc = base_model.get_layer('block_5_add').output
x_inc_out = Inception(x_inc, 32)
x3 = CondConv2D(kernel_size=3, filters=32, stride=2, padding='same', num_experts=3)(x_inc_out)
x3 = layers.Conv2D(kernel_size=4, filters=29, strides=1, padding='valid')(x3)
x_conv3 = sse_block(x3, ratio=4)

# --- 4. VISION TRANSFORMER PATH (HIGH DROPOUT) ---
patches = Patches(patch_size)(x_inc_out) 
encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)
encoded_patches = layers.Dropout(HIGH_DROPOUT)(encoded_patches) 

for _ in range(transformer_layers):
    x_norm1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=HIGH_DROPOUT)(x_norm1, x_norm1) 
    x_add1 = layers.Add()([attention_output, encoded_patches])
    x_norm2 = layers.LayerNormalization(epsilon=1e-6)(x_add1)
    x_mlp = mlp(x_norm2, hidden_units=transformer_units, dropout_rate=HIGH_DROPOUT) 
    encoded_patches = layers.Add()([x_mlp, x_add1])

x_final_trans = layers.LayerNormalization(epsilon=1e-6, name='cam_layer')(encoded_patches)
x_t = layers.Reshape((4, 4, 32))(x_final_trans) 
x_t = layers.Resizing(11, 11, interpolation='bilinear')(x_t)

# --- 5. FUSION & CLASSIFICATION ---
merged = layers.Add()([x_conv1, x_conv2, x_conv3, x_t])
merged = sse_block(merged, ratio=4)
merged = layers.GlobalAveragePooling2D()(merged)
predictions = layers.Dense(TOTAL_CLASSES, activation='softmax')(merged)

drone_model_v2 = tf.keras.Model(inputs=inputs, outputs=predictions)
print("✅ drone_model_v2 successfully built!")

# Added model summary to verify parameter count
drone_model_v2.summary()

In [ ]:
import os
import tensorflow as tf
from tqdm.keras import TqdmCallback

# 1. Compile the V2 Model
drone_model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 2. Callbacks specifically for V2
checkpoint_dir = r"/content/drive/MyDrive/plant_disease_dataset/checkpoints"
v2_checkpoint_path = os.path.join(checkpoint_dir, "best_drone_model_v2.keras")

v2_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=v2_checkpoint_path,
    monitor='val_loss',
    save_best_only=True, 
    verbose=1 
)

standard_reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

standard_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True
)

tqdm_callback = TqdmCallback(verbose=1)

print("="*60)
print("PHASE 2 & 3: REGULARIZED TRAINING INITIATED")
print("="*60)

# 3. Fit the V2 Model
history_v2 = drone_model_v2.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=20,
    class_weight=class_weight_dict, 
    verbose=0,
    callbacks=[
        v2_checkpoint_cb, 
        standard_early_stopping, 
        standard_reduce_lr, 
        tqdm_callback
    ]
)